In [97]:
from langchain_openai import ChatOpenAI
from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
from typing import List
from openai import OpenAI
import importlib
import json
import time
import tools

from importlib import reload
reload(tools)
from tools import *

In [81]:
custom_base_url = "https://aihub-api.sktelecom.com/aihub/v2/sandbox"
api_key = "e97ee307-a791-4e06-ade1-df4b9d032eed" ############ 지우고커밋!!!!!! 
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_base=custom_base_url,
    openai_api_key=api_key
)

def call_llm_with_retries(
    llm,
    messages,
    *,
    expect_json=False,
    max_retries=3,
    sleep_sec=1
):
    """
    LLM 호출을 안전하게 감싸는 함수.

    - expect_json=True: 응답을 JSON으로 파싱해서 반환
    - expect_json=False: 응답 string 그대로 반환
    """

    for attempt in range(1, max_retries + 1):
        response = llm.invoke(messages)
        raw_content = response.content.strip()
        print(f"\n[call_llm_with_retries] Attempt {attempt}:")
        # print(repr(raw_content))

        if not raw_content:
            print("⚠️ 빈 응답입니다. 재시도합니다...")
            time.sleep(sleep_sec)
            continue

        if expect_json:
            # 코드 블럭 제거
            cleaned = raw_content
            if cleaned.startswith("```"):
                cleaned = "\n".join(
                    line for line in cleaned.splitlines()
                    if not line.strip().startswith("```")
                )
            try:
                return json.loads(cleaned)
            except json.JSONDecodeError as e:
                print(f"❌ JSON 파싱 실패: {e}")
                # JSON 오류는 재시도하지 않고 바로 예외
                raise e
        else:
            return raw_content

    raise ValueError("❌ LLM이 유효한 응답을 {max_retries}회 시도했으나 받지 못했습니다.")

In [63]:
## 1. pre-planning : 쿼리 정규화 - 사용자 의도에 따라 expansion, breakdown .. 

def paraphrase_intent(user_input: str):
    prompt = f"""
        You will receive a user utterance below.
        This utterance may be very short, abstract, or a long descriptive question.
        Please perform the following tasks:
        
        1. Clearly describe the hidden intent behind the utterance.
        2. If necessary, expand the utterance to make it more specific, or if it is too long, summarize it into the core question.
        3. Summarize in one sentence what kind of final answer the user is expecting.
        
        All output must be written in Korean.
        
        Output the result in pure JSON format.
        
        Example output format:
        {{
          "intent": "데이터 사용량 부족으로 요금제 변경 의사",
          "paraphrased_query": "데이터 용량이 부족해서 더 큰 데이터 제공 요금제를 추천받고 싶습니다.",
          "expected_answer": "고객이 현재 가입 가능한 데이터 용량이 큰 요금제 목록"
        }}
        
        User utterance:
        "{user_input}"
    """

    response = call_llm_with_retries(
        llm,
        [
            {"role": "system", "content": "You are a skilled customer service assistant."},
            {"role": "user", "content": prompt}
        ],
        expect_json=True
    )
    return response

In [64]:
## 2. planning : tool-aware planning 

def plan_steps(query_metadata: dict) -> str:
    """
    Given the structured query metadata, create a step-by-step plan including tools and inputs.
    """
    intent = query_metadata["intent"]
    paraphrased_query = query_metadata["paraphrased_query"]
    expected_answer = query_metadata["expected_answer"]

    prompt = f"""
        당신은 고객 서비스 질의를 처리하는 고급 AI Assistant입니다.
        
        아래에 사용자가 질의한 의도 정보가 주어집니다.
        당신의 목표는 최종 답변을 생성하기 위한 단계별 계획을 수립하는 것입니다.
        
        각 단계는 다음 중 하나일 수 있습니다:
        - 특정 Tool을 사용하여 데이터를 검색/처리
        - Tool 없이 LLM만으로 결과를 요약/종합/답변 생성
        
        사용 가능한 도구 목록과 설명은 다음과 같습니다:
        
        [Available Tools]
        1. search_neo4j
            - 입력: 구체화된 질문 또는 요금제 관련 질의 (string)
            - 출력: Neo4j에서 검색된 데이터 요약(dict)
        
        2. get_service_info
            - 입력: 서비스관리번호 (svc_mgmt_num) (string)
            - 출력: 고객 기본정보 및 요금제 정보(dict)
        
        3. get_subscribed_products
            - 입력: 서비스관리번호 (svc_mgmt_num) (string)
            - 출력: 고객 가입 상품 목록(dict)
        
        또한, Tool을 사용하지 않고 LLM만으로 처리하는 단계도 정의할 수 있습니다.
        이 경우 "tool" 값에 "none"을 입력하세요.
        
        [규칙]
        - 단계 수는 최소 1단계에서 최대 5단계까지 자유롭게 계획하세요.
        - 반드시 사용자의 기대 결과(expected_answer)를 달성하기 위한 모든 단계를 포함해야 합니다.
        - '추천', '비교', '요약'이 필요한 경우, 마지막 단계에 결과 생성 단계를 반드시 포함해야 합니다.
        - Tool이 "none"인 단계에서는, 이전 단계 결과를 단순 요약하지 말고 **추천, 비교, 상세 안내를 포함하는 구체적인 답변 생성 지침**을 tool_input에 명시하세요.
        - 각 단계에는 어떤 tool을 사용할지와 그 tool의 입력값을 명확히 지정하세요.
        - 중복 단계 없이 효율적으로 계획하세요.
        - 반드시 검색 도구를 먼저 사용한 후, 마지막 단계에서 LLM 요약을 생성해야 합니다.
        
        [User Query Metadata]
        - Intent: {query_metadata["intent"]}
        - Paraphrased Query: {query_metadata["paraphrased_query"]}
        - Expected Answer: {query_metadata["expected_answer"]}
        
        [추가 정보 요구사항 예시]
        아래 예시를 참고해 필요한 정보 항목을 목록으로 작성하세요. 필요 시 다른 항목을 자유롭게 추가할 수 있습니다.
        예: ["요금제 가격", "데이터 용량", "부가 혜택", "계약 기간"]
        
        아래 JSON 배열 형식으로 출력하세요:
        
        [
          {{
            "step_number": 1,
            "description": "이 단계에서 무엇을 할지 설명",
            "tool": "사용할 Tool 이름 또는 'none'",
            "tool_input": "Tool에 넣을 입력값 (또는 이전 단계 출력에 기반한 구체적인 안내)",
            "required_details": ["..."]
          }},
          ...
        ]
        
        [출력 예시]
        [
          {{
            "step_number": 1,
            "description": "Neo4j에서 데이터 용량이 큰 요금제 정보를 검색",
            "tool": "search_neo4j",
            "tool_input": "{query_metadata["paraphrased_query"]}",
            "required_details": ["데이터 용량", "요금제 가격"]
          }},
          {{
            "step_number": 2,
            "description": "검색 결과를 요약하여 추천 답변을 생성",
            "tool": "none",
            "tool_input": "이전 단계 출력에 기반해 고객에게 적합한 요금제를 추천하고 각 요금제의 장단점을 설명",
            "required_details": ["추천 이유", "부가 혜택"]
          }}
        ]
    """
    response = call_llm_with_retries(
        llm,
        [
            {"role": "system", "content": "You are an expert planning assistant for customer service queries."},
            {"role": "user", "content": prompt}
        ],
        expect_json=True
    )
    return response

In [105]:
## 3. validate - 전체 step이 실행되기 전에 각 step 마다 결과에 대한 validation

def validate_step(step_input, step_output, paraphrased_query, current_step_number, total_steps, user_intent, cypher_query, raw_records):
    prompt = f"""
        당신은 Neo4j 검색 Agent의 품질 검수자입니다.
        
        아래 정보에 기반하여 현재 단계의 실행이 적절했는지 평가하세요.
        
        [평가 기준]
        - 현재 단계가 마지막 단계가 아닌 경우: 데이터 조회, 정보 수집만 수행해도 유효합니다.
        - 현재 단계가 마지막 단계라면: 사용자의 요청에 대한 최종적이고 구체적인 답변(추천, 비교, 요약)을 포함해야 합니다.
        - 쿼리 구문 오류나 명백한 실패가 있으면 실패로 간주합니다.
        
        [정보]
        - Step Number: {current_step_number}
        - Total Steps: {total_steps}
        - 사용자 의도: {user_intent}
        - Cypher 쿼리: {cypher_query}
        - 쿼리 결과 샘플: {raw_records[:10]}
        - 생성된 답변 (또는 이 단계의 출력): {step_output}
        
        아래 형식으로 JSON을 출력하세요:
        
        {{
          "is_valid": "Yes" or "No",
          "reason": "간단하고 명확한 문제 여부 설명",
          "suggestion": "문제가 있다면 개선 방안, 없다면 'None'"
        }}
    """
    print(prompt)
    response = call_llm_with_retries(
        llm,
        [
            {"role": "system", "content": "You are an answer validator."},
            {"role": "user", "content": prompt}
        ],
        expect_json=True
    )
    return response

In [110]:
## 4. replanning 포함한 전체 실행 loop --> 나중에 main.py 파일로 변경 

# 1️⃣ 유저 질문
# user_query = "데이터가 너무 빨리 닳아. 더 많은 요금제를 추천해줘."
user_query = "무제한 데이터"

# 2️⃣ paraphrase_intent 실행
metadata = paraphrase_intent(user_query)

# 3️⃣ 최초 계획 생성
plan = plan_steps(metadata)
print("=== 생성된 Plan ===")
print(plan)

# 4️⃣ 실행 루프
all_results = []
execution_count = 0
max_replans = 2   # 최대 재계획 횟수

while True:
    execution_count += 1
    print(f"\n🔄 실행 Attempt #{execution_count}")

    step_num = 0
    plan_failed = False
    step_results = []

    previous_step_output = ""
    for step in plan:
        print(f"=== Executing Step {step['step_number']} ({step['tool']}) ===")
        step_num += 1
        tool = step["tool"]
        tool_input = step["tool_input"]

        print(f"\n=== Step {step_num}: {step['description']} ===")

        # 4-1) Tool 실행
        if step["tool"] == "search_neo4j":
            output = search_neo4j(step["tool_input"], metadata)
            cypher_query_text = output["cypher_query"]
            raw_records_text = output["raw_records"]
            step_output = output["summary"]
    
        elif step["tool"] == "get_service_info":
            output = get_service_info(step["tool_input"])
            cypher_query_text = ""
            raw_records_text = ""
            step_output = str(output)
    
        elif step["tool"] == "get_subscribed_products":
            output = get_subscribed_products(step["tool_input"])
            cypher_query_text = ""
            raw_records_text = ""
            step_output = str(output)
    
        elif step["tool"] == "none":
            required_details = step.get("required_details", [])
            details_text = "\n".join(f"- {d}" for d in required_details)
            # 예: LLM으로 요약 생성
            summary_prompt = f"""
                이전 단계 출력: {previous_step_output}
                
                아래 정보를 반드시 포함해 답변을 생성하세요:
                {details_text}
                
                이 정보를 바탕으로 고객이 이해하기 쉽게 요약하고 추천하세요.
            """
            response = call_llm_with_retries(
                llm,
                [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": summary_prompt}
                ],
                expect_json=False
            )
            cypher_query_text = ""
            raw_records_text = ""
            step_output = response
    
        else:
            raise ValueError("Unknown tool")

        # 4-2) 결과 저장
        step_results.append({
            "step_number": step_num,
            "description": step["description"],
            "tool": tool,
            "input": tool_input,
            "output": output
        })

        # 4-3) Validation
        validation_data = validate_step(
            step_input=step["tool_input"],
            step_output=step_output,
            paraphrased_query=metadata["paraphrased_query"],
            current_step_number=step_num,
            total_steps=len(plan),
            user_intent=metadata["intent"],
            cypher_query=cypher_query_text,
            raw_records=raw_records_text
        )
        print(f"✅ Validation result: {validation_data}")

        if validation_data["is_valid"] == "No":
            print("⚠️ Validation failed. Replanning...")
            # paraphrased_query를 개선
            revised_query = f"{metadata['paraphrased_query']} (추가 고려사항: {validation_data['suggestion']})"
            metadata["paraphrased_query"] = revised_query

            # Replanning
            plan = plan_steps(metadata)

            # 재계획 플래그
            plan_failed = True
            break

        # ✅ 이전 단계 출력 저장
        previous_step_output = step_output
    
    # 4-4) 모든 step 성공적으로 완료
    if not plan_failed:
        all_results.extend(step_results)
        print("\n✅ 모든 단계를 성공적으로 완료했습니다.")
        break

    # 4-5) 재계획 횟수 제한
    if execution_count >= max_replans:
        print("❌ 최대 재계획 횟수를 초과했습니다. 프로세스를 종료합니다.")
        break

# 5️⃣ 결과 출력
print("\n=== 전체 실행 결과 ===")
for r in all_results:
    print(f"\nStep {r['step_number']} ({r['tool']})")
    print(f"Input: {r['input']}")
    print(f"Output: {r['output']}")


[call_llm_with_retries] Attempt 1:

[call_llm_with_retries] Attempt 1:
=== 생성된 Plan ===
[{'step_number': 1, 'description': 'Neo4j에서 무제한 데이터 요금제에 대한 정보를 검색', 'tool': 'search_neo4j', 'tool_input': '무제한 데이터 요금제에 대한 자세한 정보를 알고 싶습니다.', 'required_details': ['요금제 가격', '데이터 용량', '부가 혜택', '계약 기간']}, {'step_number': 2, 'description': '검색 결과를 기반으로 고객에게 적합한 무제한 데이터 요금제를 추천하고, 각 요금제의 장단점을 설명', 'tool': 'none', 'tool_input': '이전 단계의 결과를 활용하여 고객의 요구에 맞는 최적의 무제한 데이터 요금제 및 그 특징을 정리하여 안내', 'required_details': ['추천 요금제', '추천 이유', '장단점 설명']}]

🔄 실행 Attempt #1
=== Executing Step 1 (search_neo4j) ===

=== Step 1: Neo4j에서 무제한 데이터 요금제에 대한 정보를 검색 ===
[search_neo4j] MATCH (y:요금제)
WHERE y.상품설명 CONTAINS '무제한' OR y.마케팅키워드 CONTAINS '무제한'
RETURN y

        당신은 Neo4j 검색 Agent의 품질 검수자입니다.

        아래 정보에 기반하여 현재 단계의 실행이 적절했는지 평가하세요.

        [평가 기준]
        - 현재 단계가 마지막 단계가 아닌 경우: 데이터 조회, 정보 수집만 수행해도 유효합니다.
        - 현재 단계가 마지막 단계라면: 사용자의 요청에 대한 최종적이고 구체적인 답변(추천, 비교, 요약)을 포함해야 합니다.
        - 쿼리 구문 오류나 명백한 실패가 있으면 실패로 

In [99]:
dd = {"user_query": "무제한 데이터 요금제에 대한 자세한 정보를 알고 싶습니다.", "cypher_query": "MATCH (y:요금제)-[:제공]->(d:데이터용량)\nWHERE y.상품설명 CONTAINS '무제한' OR y.마케팅키워드 CONTAINS '무제한'\nRETURN y, d", "records": [{"y": {"라인업": "다이렉트플랜", "영문상품명": "Direct5G 69", "월정액": 69000, "상품코드매핑": ["NA00008100"], "부가세제외월정액": 62728, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["온라인", "비대면", "티다샵", "유심개통", "티다이렉트샵", "쓰던폰", "USIM개통", "데이터무제한", "자급제", "온라인전용요금제", "T다이렉트전용요금제", "T다샵", "다이렉트플랜", "스마트워치요금할인혜택", "멤버십VIP혜택", "우주패스무료혜택", "태블릿요금할인혜택", "FLO무료혜택", "콘텐츠할인", "혜택요금제", "wavve무료혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 69", "선택약정할인포함부가세제외월정액": 69000, "net가격": 55455, "운영상태": "운영", "고유ID": "PA00000010", "상품설명": "데이터를 무제한으로 이용 가능하며 T멤버십 VIP, 구독서비스 무료 등 다양한 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트플랜", "영문상품명": "Direct5G 62", "월정액": 62000, "상품코드매핑": ["NA00008104"], "부가세제외월정액": 56364, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["비대면", "데이터무제한", "온라인", "티다이렉트샵", "USIM개통", "유심개통", "자급제", "쓰던폰", "T다이렉트전용요금제", "wavve할인혜택", "다이렉트플랜", "온라인전용요금제", "티다샵", "T다샵", "멤버십VIP", "혜택요금제", "FLO할인혜택", "우주패스할인혜택", "콘텐츠할인"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 62", "선택약정할인포함부가세제외월정액": 62000, "net가격": 55455, "운영상태": "운영", "고유ID": "PA00000012", "상품설명": "데이터를 무제한으로 이용 가능하며 T멤버십 VIP, 구독서비스 할인 등 다양한 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 60.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "(구) T플랜", "영문상품명": "Data Infinity", "월정액": 100000, "상품코드매핑": ["NA00005959"], "부가세제외월정액": 90909, "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["구버전티플랜", "구버전T플랜", "데이터무제한", "데이터공유", "가족모아", "T가족모아데이터", "가족끼리데이터공유", "데이터공유요금제", "VIP멤버십제공", "데이터공유가능", "스마트워치요금무료", "태블릿요금무료", "로밍혜택", "OTT콘텐츠혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "Data 인피니티", "선택약정할인포함부가세제외월정액": 74975, "net가격": 90909, "운영상태": "가입중단", "고유ID": "PA00000036", "상품설명": "데이터를 무제한으로 제공하며 가족 간 데이터 공유가 가능한 T가족모아데이터를 이용할 수 있고 인피니티 만의 특화 혜택을 제공하는 요금제", "통신규격": ["3G generation", "LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜", "영문상품명": "(구)5GX프라임", "상품코드매핑": ["NA00006404"], "월정액": 89000, "부가세제외월정액": 80909, "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["wavve혜택", "5GX플랜", "Wavve", "스트리밍", "OTT혜택", "혜택요금제", "FLO70%할인", "wavve70%할인", "우주패스5천원할인", "태블릿요금무료혜택", "스마트워치요금무료혜택", "VIP멤버십혜택", "FLO할인혜택", "우주패스할인혜택", "제휴할인혜택", "데이터무제한"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "(구)5GX프라임", "선택약정할인포함부가세제외월정액": 66725, "net가격": 86364, "운영상태": "운영", "고유ID": "PA00000041", "상품설명": "무제한 데이터와 T멤버십 VIP, 구독서비스 할인 등을 제공하는 혜택 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 60.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜", "영문상품명": "(구)5GX플래티넘", "상품코드매핑": ["NA00006405"], "월정액": 125000, "부가세제외월정액": 113637, "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["FLO무료혜택", "우주패스무료혜택", "제휴할인혜택", "데이터무제한", "혜택요금제", "최다혜택요금제", "스트리밍", "Wavve", "VIP멤버십혜택", "스마트워치요금무료혜택", "태블릿요금무료혜택", "5GX플랜", "wavve혜택", "혜택요금제", "OTT혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "(구)5GX플래티넘", "선택약정할인포함부가세제외월정액": 93705, "net가격": 113637, "고유ID": "PA00000042", "운영상태": "운영", "상품설명": "무제한 데이터와 T멤버십 VIP, 구독서비스 무료 등을 제공하는 혜택 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "T플랜", "영문상품명": "T Plan Max", "상품코드매핑": ["NA00006539"], "월정액": 100000, "부가세제외월정액": 90909, "청구방법": "후불", "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "마케팅키워드": ["T플랜", "가족모아데이터혜택", "데이터무제한", "T멤버십VIP혜택", "Wavve", "FLO", "wavve무료", "FLO무료", "가족", "가족간데이터공유혜택요금제", "가족공유혜택", "태블릿요금무료", "스마트워치요금무료", "콘텐츠무료", "인피니티"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "T플랜 맥스", "선택약정할인포함부가세제외월정액": 74975, "net가격": 90909, "운영상태": "운영", "고유ID": "PA00000048", "상품설명": "데이터를 무제한으로 제공하며 가족 간 데이터 공유가 가능한 T가족모아데이터를 이용할 수 있고 T멤버십 VIP, 구독서비스 무료 등 혜택을 제공하는 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜", "영문상품명": "5GX Platinum", "상품코드매핑": ["NA00007789"], "월정액": 125000, "부가세제외월정액": 113637, "청구방법": "후불", "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "마케팅키워드": ["OTT혜택", "혜택요금제", "wavve혜택", "5GX플랜", "Wavve", "스트리밍", "데이터무제한", "혜택요금제", "최다혜택요금제", "태블릿요금무료혜택", "스마트워치요금무료혜택", "VIP멤버십혜택", "FLO무료혜택", "우주패스무료혜택", "제휴할인혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 플래티넘", "선택약정할인포함부가세제외월정액": 93705, "net가격": 113637, "운영상태": "운영", "고유ID": "PA00000060", "상품설명": "무제한 데이터와 T멤버십 VIP, 구독서비스 무료 등을 제공하는 혜택 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜", "영문상품명": "5GX Prime", "상품코드매핑": ["NA00007790"], "월정액": 89000, "부가세제외월정액": 80909, "청구방법": "후불", "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "마케팅키워드": ["Wavve", "스트리밍", "OTT혜택", "혜택요금제", "wavve혜택", "5GX플랜", "태블릿요금무료혜택", "FLO할인혜택", "우주패스할인혜택", "제휴할인혜택", "데이터무제한", "혜택요금제", "FLO70%할인", "wavve70%할인", "우주패스5천원할인", "스마트워치요금무료혜택", "VIP멤버십혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프라임", "선택약정할인포함부가세제외월정액": 66725, "net가격": 80909, "운영상태": "운영", "고유ID": "PA00000061", "상품설명": "무제한 데이터와 T멤버십 VIP, 구독서비스 할인 등을 제공하는 혜택 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 60.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년", "영문상품명": "0 Youth 89", "상품코드매핑": ["NA00008140"], "월정액": 89000, "부가세제외월정액": 80909, "청구방법": "후불", "상품가입조건": "만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["커피50%", "wavve할인혜택", "FLO할인혜택", "우주패스할인혜택", "멤버십VIP", "혜택요금제", "콘텐츠할인", "만34세이하", "커피", "영화", "로밍", "로밍쿠폰", "로밍할인", "커피쿠폰", "커피할인", "영화50%", "영화쿠폰", "영화할인", "baro50%", "데이터무제한"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 89", "선택약정할인포함부가세제외월정액": 66725, "net가격": 80909, "고유ID": "PA00000065", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택을 제공하는 만 34세 이하 개인 고객만 가입 가능한 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "넷플릭스 요금제", "영문상품명": "5GX Prime Plus(Netflix)", "상품코드매핑": ["NA00008721"], "월정액": 99000, "부가세제외월정액": 90000, "청구방법": "후불", "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "마케팅키워드": ["넷플릭스", "우주패스Netflix", "Netflix", "넷플릭스할인", "5GX플랜", "혜택요금제", "wavve혜택", "OTT혜택", "넷플릭스혜택", "Wavve", "스트리밍", "태블릿요금무료혜택", "스마트워치요금무료혜택", "VIP멤버십혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프라임플러스(넷플릭스)", "선택약정할인포함부가세제외월정액": 74250, "net가격": 90000, "고유ID": "PA00000066", "운영상태": "운영", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "넷플릭스 요금제", "월정액": 89000, "상품코드매핑": ["NA00008722"], "영문상품명": "5GX Prime(Netflix)", "부가세제외월정액": 80909, "청구방법": "후불", "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "마케팅키워드": ["Wavve", "스트리밍", "Netflix", "넷플릭스", "우주패스Netflix", "혜택요금제", "wavve혜택", "OTT혜택", "넷플릭스혜택", "태블릿요금할인혜택", "스마트워치요금할인혜택", "콘텐츠할인", "우주패스할인", "5GX플랜", "FLO할인", "넷플릭스할인", "VIP멤버십혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프라임(넷플릭스)", "선택약정할인포함부가세제외월정액": 66725, "net가격": 80909, "운영상태": "운영", "고유ID": "PA00000067", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 60.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 넷플릭스 요금제", "영문상품명": "0 Youth 99(Netflix)", "상품코드매핑": ["NA00008727"], "월정액": 99000, "부가세제외월정액": 90000, "청구방법": "후불", "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["로밍", "데이터무제한", "커피", "영화", "만34세이하", "커피쿠폰", "커피50%", "커피", "로밍", "로밍쿠폰", "영화할인", "baro50%", "영화", "영화쿠폰", "커피할인", "영화50%", "로밍할인", "우주패스Netflix", "Netflix", "wavve혜택", "넷플릭스혜택", "혜택요금제", "스트리밍", "OTT혜택", "넷플릭스", "Wavve"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 99(넷플릭스)", "선택약정할인포함부가세제외월정액": 74250, "net가격": 90000, "운영상태": "운영", "고유ID": "PA00000068", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 넷플릭스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 넷플릭스 요금제", "영문상품명": "0 Youth 89(Netflix)", "상품코드매핑": ["NA00008728"], "월정액": 89000, "부가세제외월정액": 80909, "청구방법": "후불", "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["데이터무제한", "커피50%", "영화", "로밍", "만34세이하", "커피", "영화50%", "영화", "커피쿠폰", "커피할인", "커피", "로밍쿠폰", "baro50%", "로밍", "영화쿠폰", "영화할인", "Wavve", "스트리밍", "Netflix", "넷플릭스", "로밍할인", "우주패스Netflix", "혜택요금제", "wavve혜택", "OTT혜택", "넷플릭스혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 89(넷플릭스)", "선택약정할인포함부가세제외월정액": 66725, "net가격": 80909, "운영상태": "운영", "고유ID": "PA00000069", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 넷플릭스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "넷플릭스 요금제", "영문상품명": "5GX Premium(Netflix)", "상품코드매핑": ["NA00008720"], "월정액": 109000, "부가세제외월정액": 99091, "청구방법": "후불", "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "마케팅키워드": ["Netflix", "넷플릭스", "우주패스Netflix", "OTT혜택", "넷플릭스혜택", "Wavve", "스트리밍", "스마트워치요금무료혜택", "VIP멤버십혜택", "5GX플랜", "태블릿요금무료혜택", "wavve혜택", "넷플릭스무료", "혜택요금제"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프리미엄(넷플릭스)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "고유ID": "PA00000071", "운영상태": "운영", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트 넷플릭스 요금제", "영문상품명": "Direct 5G 69(Netflix)", "상품코드매핑": ["NA00008724"], "월정액": 69000, "부가세제외월정액": 62728, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["온라인", "비대면", "데이터무제한", "T다샵", "우주패스Netflix", "티다이렉트샵", "티다샵", "USIM개통", "유심개통", "자급제", "쓰던폰", "Netflix", "넷플릭스", "Wavve", "다이렉트 넷플릭스", "T다이렉트전용요금제", "wavve혜택", "온라인전용요금제", "넷플릭스혜택", "혜택요금제", "스트리밍", "OTT혜택", "태블릿요금무료혜택", "스마트워치요금무료혜택", "넷플릭스할인", "VIP멤버십혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 69(넷플릭스)", "선택약정할인포함부가세제외월정액": 69000, "net가격": 62728, "고유ID": "PA00000072", "운영상태": "운영", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트 넷플릭스 요금제", "영문상품명": "Direct 5G 62(Netflix)", "상품코드매핑": ["NA00008725"], "월정액": 62000, "부가세제외월정액": 56364, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["온라인", "티다샵", "태블릿요금무료혜택", "콘텐츠할인", "자급제", "혜택요금제", "티다이렉트샵", "비대면", "데이터무제한", "다이렉트 넷플릭스", "넷플릭스혜택", "쓰던폰", "스트리밍", "스마트워치요금무료혜택", "유심개통", "우주패스할인", "우주패스Netflix", "온라인전용요금제", "T다샵", "OTT혜택", "Netflix", "FLO할인", "Wavve", "VIP멤버십혜택", "USIM개통", "T다이렉트전용요금제", "넷플릭스할인", "넷플릭스", "wavve혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 62(넷플릭스)", "선택약정할인포함부가세제외월정액": 62000, "net가격": 56364, "고유ID": "PA00000073", "운영상태": "운영", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 60.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 다이렉트 넷플릭스 요금제", "영문상품명": "0 Youth Direct 69(Netflix)", "상품코드매핑": ["NA00008730"], "월정액": 69000, "부가세제외월정액": 62728, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 34세 이하 개인 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["비대면", "온라인", "티다샵", "T다샵", "유심개통", "티다이렉트샵", "쓰던폰", "USIM개통", "데이터무제한", "자급제", "우주패스Netflix", "Netflix", "넷플릭스", "온라인전용요금제", "다이렉트 넷플릭스", "혜택요금제", "wavve혜택", "OTT혜택", "넷플릭스혜택", "Wavve", "스트리밍", "T다이렉트전용요금제", "넷플릭스할인", "만34세이하", "커피할인", "영화50%", "커피", "커피쿠폰", "로밍혜택", "커피50%", "커피혜택", "영화혜택", "영화쿠폰", "영화할인", "영화", "로밍쿠폰", "로밍할인", "baro50%", "로밍"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 다이렉트 69(넷플릭스)", "선택약정할인포함부가세제외월정액": 69000, "net가격": 62728, "고유ID": "PA00000074", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 19세 이상 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 다이렉트 넷플릭스 요금제", "영문상품명": "0 Youth Direct 62(Netflix)", "상품코드매핑": ["NA00008731"], "월정액": 62000, "부가세제외월정액": 56364, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 34세 이하 개인 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["데이터무제한", "온라인", "비대면", "USIM개통", "유심개통", "자급제", "쓰던폰", "Netflix", "넷플릭스", "T다샵", "우주패스Netflix", "티다이렉트샵", "티다샵", "넷플릭스혜택", "혜택요금제", "스트리밍", "OTT혜택", "Wavve", "넷플릭스할인", "다이렉트 넷플릭스", "T다이렉트전용요금제", "wavve혜택", "온라인전용요금제", "커피50%", "커피", "영화혜택", "로밍혜택", "만34세이하", "커피혜택", "영화50%", "영화", "커피쿠폰", "커피할인", "로밍할인", "로밍", "로밍쿠폰", "영화할인", "baro50%", "영화쿠폰"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 다이렉트 62(넷플릭스)", "선택약정할인포함부가세제외월정액": 62000, "net가격": 56364, "고유ID": "PA00000075", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 19세 이상 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년", "영문상품명": "0 Youth 109", "상품코드매핑": ["NA00008676"], "월정액": 109000, "부가세제외월정액": 99091, "청구방법": "후불", "상품가입조건": "만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["로밍", "로밍쿠폰", "로밍할인", "만34세이하", "멤버십VIP", "스마트워치요금무료", "영화", "YoutubePremium혜택", "단말보험할인", "데이터무제한", "5GX플랜", "baro50%", "영화쿠폰", "영화할인", "유튜브무료제공요금제", "유튜브요금제", "유튜브혜택", "커피", "커피50%", "커피쿠폰", "영화50%", "태블릿요금무료", "혜택요금제", "커피할인"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 109", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "운영상태": "운영", "고유ID": "PA00000076", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 유튜브 프리미엄을 제공하는 만 34세 이하 개인 고객만 가입 가능한 유튜브 혜택 특화 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트플랜", "영문상품명": "Direct5G 76", "월정액": 76000, "상품코드매핑": ["NA00008685"], "부가세제외월정액": 69091, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["온라인", "티다이렉트샵", "티다샵", "USIM개통", "유심개통", "자급제", "쓰던폰", "비대면", "데이터무제한", "T다샵", "다이렉트플랜", "온라인전용요금제", "스마트워치기기할인", "태블릿기기할인", "유튜브무료제공요금제", "혜택요금제", "유튜브혜택", "YoutubePremium혜택", "T다이렉트전용요금제", "유튜브요금제", "스마트폰기기할인"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 76", "선택약정할인포함부가세제외월정액": 76000, "net가격": 69091, "운영상태": "운영", "고유ID": "PA00000077", "상품설명": "데이터를 무제한으로 이용 가능하며 유튜브 프리미엄 혜택 및 T멤버십 VIP, 구독서비스 무료 등 다양한 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트 넷플릭스 요금제", "영문상품명": "Direct 5G 76(Netflix)", "상품코드매핑": ["NA00008723"], "월정액": 76000, "부가세제외월정액": 69091, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["넷플릭스혜택", "혜택요금제", "스트리밍", "OTT혜택", "넷플릭스", "Wavve", "Netflix", "다이렉트 넷플릭스", "온라인전용요금제", "태블릿기기할인", "스마트폰기기할인", "VIP멤버십혜택", "스마트워치기기할인", "태블릿요금무료혜택", "스마트워치요금무료혜택", "T다이렉트전용요금제", "넷플릭스무료", "온라인", "비대면", "티다샵", "T다샵", "유심개통", "티다이렉트샵", "쓰던폰", "USIM개통", "데이터무제한", "자급제", "우주패스Netflix", "wavve혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 76(넷플릭스)", "선택약정할인포함부가세제외월정액": 76000, "net가격": 69091, "고유ID": "PA00000078", "운영상태": "운영", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 넷플릭스 요금제", "영문상품명": "0 Youth 109(Netflix)", "상품코드매핑": ["NA00008726"], "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["영화", "로밍", "만34세이하", "커피", "커피할인", "영화50%", "커피", "커피쿠폰", "데이터무제한", "커피50%", "baro50%", "로밍", "영화쿠폰", "영화할인", "영화", "넷플릭스", "우주패스Netflix", "Netflix", "로밍쿠폰", "로밍할인", "혜택요금제", "wavve혜택", "OTT혜택", "넷플릭스혜택", "Wavve", "스트리밍"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 109(넷플릭스)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "고유ID": "PA00000079", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 넷플릭스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜", "영문상품명": "5GX Primeplus", "상품코드매핑": ["NA00007622"], "월정액": 99000, "부가세제외월정액": 90000, "청구방법": "후불", "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "마케팅키워드": ["혜택요금제", "wavve혜택", "5GX플랜", "태블릿요금무료혜택", "스마트워치요금무료혜택", "VIP멤버십혜택", "FLO무료혜택", "Wavve", "스트리밍", "OTT혜택", "혜택요금제", "우주패스무료혜택", "제휴할인혜택", "데이터무제한"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프라임플러스", "선택약정할인포함부가세제외월정액": 74250, "net가격": 90000, "운영상태": "운영", "고유ID": "PA00000080", "상품설명": "무제한 데이터와 T멤버십 VIP, 구독서비스 무료 등을 제공하는 혜택 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜", "영문상품명": "5GX Premium", "상품코드매핑": ["NA00008565"], "월정액": 109000, "부가세제외월정액": 99091, "청구방법": "후불", "상품가입조건": "5G/LTE 휴대폰 이용 고객 가입 가능", "마케팅키워드": ["YoutubePremium혜택", "5GX플랜", "스마트워치기기할인", "멤버십VIP", "데이터무제한", "단말보험할인", "유튜브요금제", "유튜브무료제공요금제", "스마트폰기기할인", "스마트워치요금무료", "태블릿기기할인", "유튜브혜택", "혜택요금제", "태블릿요금무료"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프리미엄", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "운영상태": "운영", "고유ID": "PA00000083", "상품설명": "무제한 데이터와 유튜브 프리미엄 혜택을 제공하는 유튜브 혜택 특화 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 다이렉트플랜", "월정액": 69000, "상품코드매핑": ["NA00008154"], "영문상품명": "0 Youth direct 69", "부가세제외월정액": 62728, "청구방법": "후불", "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["비대면", "데이터무제한", "자급제", "쓰던폰", "USIM개통", "유심개통", "티다이렉트샵", "티다샵", "온라인", "T다샵", "다이렉트플랜", "온라인전용요금제", "T다이렉트전용요금제", "FLO무료혜택", "우주패스무료혜택", "태블릿요금할인혜택", "스마트워치요금할인혜택", "멤버십VIP혜택", "혜택요금제", "콘텐츠할인", "만34세이하", "커피", "영화", "로밍", "커피50%", "커피쿠폰", "커피할인", "영화50%", "영화쿠폰", "영화할인", "baro50%", "로밍쿠폰", "로밍할인", "wavve무료혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 다이렉트 69", "선택약정할인포함부가세제외월정액": 69000, "net가격": 62728, "운영상태": "운영", "고유ID": "PA00000090", "상품설명": "무제한 데이터와 0청년 특화 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 다이렉트플랜", "월정액": 62000, "상품코드매핑": ["NA00008155"], "영문상품명": "0 Youth direct 62", "부가세제외월정액": 56364, "청구방법": "후불", "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["자급제", "온라인", "비대면", "데이터무제한", "wavve할인혜택", "FLO할인혜택", "쓰던폰", "USIM개통", "유심개통", "티다이렉트샵", "티다샵", "T다샵", "다이렉트플랜", "온라인전용요금제", "T다이렉트전용요금제", "커피50%", "커피쿠폰", "우주패스할인혜택", "멤버십VIP", "혜택요금제", "콘텐츠할인", "만34세이하", "커피", "영화", "로밍", "로밍쿠폰", "로밍할인", "커피할인", "영화50%", "영화쿠폰", "영화할인", "baro50%"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 다이렉트 62", "선택약정할인포함부가세제외월정액": 62000, "net가격": 56364, "운영상태": "운영", "고유ID": "PA00000091", "상품설명": "무제한 데이터와 0청년 특화 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년", "영문상품명": "0 Youth 99", "상품코드매핑": ["NA00008139"], "월정액": 99000, "부가세제외월정액": 90000, "청구방법": "후불", "상품가입조건": "만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "마케팅키워드": ["FLO무료혜택", "우주패스무료혜택", "태블릿요금할인혜택", "스마트워치요금할인혜택", "멤버십VIP혜택", "혜택요금제", "콘텐츠할인", "만34세이하", "커피", "영화", "로밍", "데이터무제한", "커피50%", "커피쿠폰", "커피할인", "영화50%", "영화쿠폰", "영화할인", "baro50%", "로밍쿠폰", "로밍할인", "wavve무료혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 99", "선택약정할인포함부가세제외월정액": 74250, "net가격": 90000, "고유ID": "PA00000106", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택을 제공하는 만 34세 이하 개인 고객만 가입 가능한 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "넷플릭스 요금제", "영문상품명": "5GX Platinum(Netflix)", "상품코드매핑": ["NA00008719"], "월정액": 125000, "부가세제외월정액": 113637, "청구방법": "후불", "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "마케팅키워드": ["wavve혜택", "넷플릭스", "넷플릭스무료", "넷플릭스혜택", "스마트워치요금무료혜택", "5GX플랜", "Netflix", "OTT혜택", "VIP멤버십혜택", "Wavve", "스트리밍", "우주패스Netflix", "태블릿요금무료혜택", "혜택요금제"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 플래티넘(넷플릭스)", "선택약정할인포함부가세제외월정액": 93705, "net가격": 113637, "고유ID": "PA00000231", "운영상태": "운영", "상품설명": "무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 0.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 120.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "null", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜 스마트기기 요금제", "영문상품명": "5GX Platinum(smart device)", "상품코드매핑": ["NA00009099"], "월정액": 125000, "부가세제외월정액": 113637, "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "청구방법": "후불", "마케팅키워드": ["스마트기기", "VIP멤버십혜택", "스트리밍", "wavve혜택", "Wavve", "애플워치", "갤럭시워치", "데이터무제한", "5GX플랜", "태블릿요금무료혜택", "스마트워치요금무료혜택", "혜택요금제"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 플래티넘(스마트기기)", "선택약정할인포함부가세제외월정액": 125000, "net가격": 113637, "고유ID": "PA00000693", "운영상태": "운영", "상품설명": "무제한 데이터와 스마트 기기 device 할인 멤버십을 제공하는 스마트 기기 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜 디즈니+ 요금제", "영문상품명": "5GX Prime Plus(Disney+)", "상품코드매핑": ["NA00009126"], "월정액": 99000, "부가세제외월정액": 90000, "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "청구방법": "후불", "마케팅키워드": ["5GX플랜", "혜택요금제", "wavve혜택", "디즈니할인", "디즈니혜택", "디즈니", "Disney", "OTT혜택", "스마트워치요금무료혜택", "태블릿요금무료혜택", "스트리밍", "디즈니플러스", "VIP멤버십혜택", "Wavve"], "상품분류": "상품 > 기본요금제", "상품명": "5GX 프라임플러스(디즈니+)", "선택약정할인포함부가세제외월정액": 74250, "net가격": 90000, "고유ID": "PA00002803", "운영상태": "운영", "상품설명": "무제한 데이터와 디즈니 플러스 멤버십을 제공하는 디즈니 플러스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트5G 유튜브 프리미엄 요금제", "상품코드매핑": ["NA00009122"], "영문상품명": "Direct5G 76(youtube premium)", "월정액": 76000, "부가세제외월정액": 69091, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["티다이렉트샵", "스마트폰기기할인", "쓰던폰", "티다샵", "온라인", "유튜브혜택", "스마트워치요금무료혜택", "유심개통", "YoutubePremium혜택", "데이터무제한", "비대면", "스트리밍", "혜택요금제", "Wavve", "T다이렉트전용요금제", "VIP멤버십혜택", "스마트워치기기할인", "유튜브요금제", "자급제", "태블릿기기할인", "wavve혜택", "유튜브무료제공요금제", "USIM개통", "T다샵", "온라인전용요금제", "태블릿요금무료혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 76(유튜브 프리미엄)", "선택약정할인포함부가세제외월정액": 76000, "net가격": 69091, "운영상태": "운영", "고유ID": "PA00002804", "상품설명": "무제한 데이터와 유튜브 프리미엄 멤버십을 제공하는 유튜브 프리미엄 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 디즈니+ 요금제", "영문상품명": "0 Youth 99(Disney+)", "상품코드매핑": ["NA00009130"], "월정액": 99000, "부가세제외월정액": 90000, "청구방법": "후불", "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)\t", "마케팅키워드": ["영화할인", "영화쿠폰", "baro50%", "커피", "디즈니플러스", "로밍할인", "디즈니", "데이터무제한", "OTT혜택", "디즈니혜택", "만34세이하", "스트리밍", "Wavve", "커피50%", "영화", "커피쿠폰", "로밍쿠폰", "혜택요금제", "wavve혜택", "커피할인", "로밍", "영화50%", "Disney"], "상품분류": "상품 > 기본요금제", "상품명": "0 청년 99(디즈니+)", "선택약정할인포함부가세제외월정액": 74250, "net가격": 90000, "운영상태": "운영", "고유ID": "PA00002805", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 디즈니 플러스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 디즈니 플러스 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트5G 디즈니+ 요금제", "영문상품명": "Direct5G 76(Disney+)", "상품코드매핑": ["NA00009127"], "월정액": 76000, "부가세제외월정액": 69091, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["T다샵"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 76(디즈니+)", "선택약정할인포함부가세제외월정액": 76000, "net가격": 69091, "고유ID": "PA00002806", "운영상태": "운영", "상품설명": "무제한 데이터와 디즈니 플러스 멤버십을 제공하는 디즈니 플러스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 디즈니+ 요금제", "영문상품명": "0 Youth 109(Disney+)", "상품코드매핑": ["NA00009129"], "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["로밍쿠폰", "영화쿠폰", "디즈니", "커피할인", "데이터무제한", "로밍할인", "Disney", "커피", "OTT할인", "영화", "baro50%", "혜택요금제", "커피50%", "Wavve", "wavve혜택", "만34세이하", "영화50%", "커피쿠폰", "영화할인", "스트리밍", "로밍", "디즈니플러스"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 109(디즈니+)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 107920, "고유ID": "PA00002807", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 디즈니 플러스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 디즈니 플러스 전용 요금제", "통신규격": []}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜 유튜브 프리미엄 요금제", "상품코드매핑": ["NA00009121"], "영문상품명": "5GX Premium(Youtube Premium)", "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "청구방법": "후불", "마케팅키워드": ["태블릿기기할인", "유튜브혜택", "단말보험할인", "유튜브요금제", "태블릿요금무료", "혜택요금제", "5GX플랜", "스마트워치요금무료", "스마트워치기기할인", "유튜브무료제공요금제", "데이터무제한", "스마트폰기기할인", "멤버십VIP", "YoutubePremium혜택"], "상품분류": "상품 > 기본요금제", "상품명": "5GX 프리미엄(유튜브 프리미엄)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "고유ID": "PA00002808", "운영상태": "운영", "상품설명": "무제한 데이터와 유튜브 프리미엄멤버십을 제공하는 유튜브 혜택 특화 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜 디즈니+ 요금제", "영문상품명": "5GX Premium(Disney+)", "상품코드매핑": ["NA00009125"], "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "청구방법": "후불", "마케팅키워드": ["단말보험할인", "Disney", "데이터무제한", "5GX플랜", "멤버십VIP", "스마트폰기기할인", "스마트워치요금무료", "태블릿기기할인", "디즈니", "태블릿요금무료", "디즈니플러스", "혜택요금제", "스마트워치기기할인"], "상품분류": "상품 > 기본요금제", "상품명": "5GX 프리미엄(디즈니+)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "고유ID": "PA00002809", "운영상태": "운영", "상품설명": "무제한 데이터와 디즈니 플러스 멤버십을 제공하는 디즈니 플러스 전용 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "5GX플랜 스마트기기 요금제", "영문상품명": "5GX Premium (Smart Device)", "상품코드매핑": ["NA00009100"], "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "19세 이상 성인, 고객 명의당 1개만 가입 가능", "청구방법": "후불", "마케팅키워드": ["태블릿요금무료", "스마트워치기기할인", "스마트폰기기할인", "스마트워치요금무료", "데이터무제한", "단말보험할인", "혜택요금제", "태블릿기기할인", "스마트기기", "5GX플랜", "애플워치", "갤럭시워치", "멤버십VIP"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "5GX 프리미엄(스마트기기)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "고유ID": "PA00002810", "운영상태": "운영", "상품설명": "무제한 데이터와 스마트 기기 device 할인 멤버십을 제공하는 스마트 기기 전용 요금제", "통신규격": ["5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 스마트기기 요금제", "영문상품명": "0 Youth 109(Smart device)", "상품코드매핑": ["NA00009102"], "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["Wavve", "로밍할인", "스마트기기", "커피", "갤럭시워치", "커피50%", "혜택요금제", "스트리밍", "영화50%", "애플워치", "영화", "baro50%", "커피할인", "커피쿠폰", "데이터무제한", "영화쿠폰", "로밍쿠폰", "로밍", "wavve혜택", "영화할인", "만34세이하"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 109(스마트기기)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "고유ID": "PA00002811", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 스마트 기기 device 할인을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 스마트 기기 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 유튜브 프리미엄 요금제", "상품코드매핑": ["NA00009123"], "영문상품명": "0 Youth 109(Youtube premium)", "월정액": 109000, "부가세제외월정액": 99091, "상품가입조건": "만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["로밍할인", "데이터무제한", "커피할인", "영화50%", "유튜브요금제", "유튜브혜택", "영화", "커피쿠폰", "Wavve", "커피", "wavve혜택", "baro50%", "영화쿠폰", "YoutubePremium혜택", "혜택요금제", "로밍", "스트리밍", "로밍쿠폰", "영화할인", "만34세이하", "유튜브무료제공요금제", "커피50%"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 109(유튜브 프리미엄)", "선택약정할인포함부가세제외월정액": 81720, "net가격": 99091, "운영상태": "운영", "고유ID": "PA00002812", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 유튜브 프리미엄 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 유튜브 프리미엄 전용 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 120.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트5G 스마트기기 요금제", "영문상품명": "Direct5G 76(smart device)", "상품코드매핑": ["NA00009101"], "월정액": 76000, "부가세제외월정액": 69091, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["쓰던폰", "유심개통", "태블릿기기할인", "자급제", "온라인전용요금제", "티다이렉트샵", "Wavve", "스트리밍", "티다샵", "VIP멤버십혜택", "T다이렉트전용요금제", "스마트기기", "태블릿요금무료혜택", "T다샵", "비대면", "혜택요금제", "애플워치", "스마트워치요금무료혜택", "스마트폰기기할인", "데이터무제한", "갤럭시워치", "스마트워치기기할인", "USIM개통", "온라인", "wavve혜택"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트5G 76(스마트기기)", "선택약정할인포함부가세제외월정액": 76000, "net가격": 69091, "고유ID": "PA00002813", "운영상태": "운영", "상품설명": "무제한 데이터와 스마트 기기 device 할인 멤버십을 제공하는 스마트 기기 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation", "5G generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "다이렉트5G 디즈니+ 요금제", "영문상품명": "Direct 5G 69(Disney+)", "상품코드매핑": ["NA00009128"], "월정액": 69000, "부가세제외월정액": 62728, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능", "청구방법": "후불", "마케팅키워드": ["VIP멤버십혜택", "자급제", "유심개통", "우주패스", "T다샵", "쓰던폰", "디즈니혜택", "태블릿요금무료혜택", "티다샵", "스트리밍", "스마트워치요금무료혜택", "혜택요금제", "T다이렉트전용요금제", "디즈니", "비대면", "티다이렉트샵", "디즈니플러스", "USIM개통", "온라인전용요금제", "OTT혜택", "Disney", "데이터무제한", "온라인", "wavve혜택", "Wavve"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "다이렉트 5G 69(디즈니+)", "선택약정할인포함부가세제외월정액": 69000, "net가격": 55455, "고유ID": "PA00002814", "운영상태": "운영", "상품설명": "무제한 데이터와 디즈니 플러스 멤버십을 제공하는 디즈니 플러스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 80.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}, {"y": {"라인업": "0청년 다이렉트 디즈니+ 요금제", "영문상품명": "0 Youth Direct 69(Disney+)", "상품코드매핑": ["NA00009131"], "월정액": 69000, "부가세제외월정액": 62728, "상품가입조건": "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 34세 이하 개인 고객 가입 가능(고객당 1번만 가입 가능)", "청구방법": "후불", "마케팅키워드": ["로밍혜택", "T다샵", "커피", "디즈니", "OTT혜택", "스트리밍", "넷플릭스할인", "Disney", "wavve혜택", "커피혜택", "커피50%", "티다이렉트샵", "영화할인", "쓰던폰", "영화쿠폰", "T다이렉트전용요금제", "로밍", "자급제", "유심개통", "데이터무제한", "혜택요금제", "온라인전용요금제", "커피쿠폰", "로밍할인", "영화혜택", "티다샵", "커피할인", "Wavve", "디즈니플러스", "USIM개통", "baro50%", "로밍쿠폰", "영화", "영화50%", "만34세이하", "비대면", "온라인"], "상품분류": "상품 > 기본요금제 > 휴대폰 요금제", "상품명": "0 청년 다이렉트 69(디즈니+)", "선택약정할인포함부가세제외월정액": 69000, "net가격": 62728, "고유ID": "PA00002815", "운영상태": "운영", "상품설명": "무제한 데이터와 0청년 특화 혜택 외 디즈니 플러스 멤버십을 제공하는 디즈니 플러스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 19세 이상 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제", "통신규격": ["5G generation", "LTE generation"]}, "d": {"최대데이터선물가능용량": 2.0, "데이터소진후데이터제공속도": 0.0, "기본제공데이터중mvoip용량": 99999.0, "데이터리필가능용량": 99999.0, "기본제공데이터중공유가능용량": 100.0, "데이터소진후최대금액및속도제한적용": "N", "데이터리필쿠폰선물가능여부": "Y", "데이터선물받기가능여부": "Y", "시니어대상데이터소진후최대금액및속도제한적용": "N", "기본제공데이터용량": 99999.0}}], "summary_prompt": "\n        아래에 Neo4j 쿼리 결과 샘플 20건이 있습니다.\n        \n        이 샘플을 참고하여 **사용자의 질문 의도와 기대하는 답변에 부합하는 상세한 설명과 추천**을 생성하세요.\n        \n        아래 정보에 반드시 기반하세요:\n        - User Intent: 무제한 데이터 요금제에 대한 정보 요청\n        - Paraphrased Query: 무제한 데이터 요금제에 대한 자세한 정보를 알고 싶습니다. (추가 고려사항: 마지막 단계에서 사용자 요청에 대한 종합적인 요약 및 추천을 포함해야 합니다.)\n        - Expected Answer: 고객이 선택할 수 있는 무제한 데이터 요금제의 상세 내용\n        \n        질문에 맞는 요금제의 특징, 가격, 데이터 용량, 가입 조건을 구체적으로 설명하고, \n        유사 요금제를 비교하며, 사용자가 쉽게 선택할 수 있도록 추천과 결론까지 포함하세요.\n        \n        (전체 데이터가 너무 커서 일부만 제공됩니다.)\n        \n        User query:\n        무제한 데이터 요금제에 대한 자세한 정보를 알고 싶습니다.\n        \n        Cypher Query:\n        MATCH (y:요금제)-[:제공]->(d:데이터용량)\nWHERE y.상품설명 CONTAINS '무제한' OR y.마케팅키워드 CONTAINS '무제한'\nRETURN y, d\n        \n        Query Result Sample:\n        [{'y': {'라인업': '다이렉트플랜', '영문상품명': 'Direct5G 69', '월정액': 69000, '상품코드매핑': ['NA00008100'], '부가세제외월정액': 62728, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['온라인', '비대면', '티다샵', '유심개통', '티다이렉트샵', '쓰던폰', 'USIM개통', '데이터무제한', '자급제', '온라인전용요금제', 'T다이렉트전용요금제', 'T다샵', '다이렉트플랜', '스마트워치요금할인혜택', '멤버십VIP혜택', '우주패스무료혜택', '태블릿요금할인혜택', 'FLO무료혜택', '콘텐츠할인', '혜택요금제', 'wavve무료혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '다이렉트5G 69', '선택약정할인포함부가세제외월정액': 69000, 'net가격': 55455, '운영상태': '운영', '고유ID': 'PA00000010', '상품설명': '데이터를 무제한으로 이용 가능하며 T멤버십 VIP, 구독서비스 무료 등 다양한 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '다이렉트플랜', '영문상품명': 'Direct5G 62', '월정액': 62000, '상품코드매핑': ['NA00008104'], '부가세제외월정액': 56364, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['비대면', '데이터무제한', '온라인', '티다이렉트샵', 'USIM개통', '유심개통', '자급제', '쓰던폰', 'T다이렉트전용요금제', 'wavve할인혜택', '다이렉트플랜', '온라인전용요금제', '티다샵', 'T다샵', '멤버십VIP', '혜택요금제', 'FLO할인혜택', '우주패스할인혜택', '콘텐츠할인'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '다이렉트5G 62', '선택약정할인포함부가세제외월정액': 62000, 'net가격': 55455, '운영상태': '운영', '고유ID': 'PA00000012', '상품설명': '데이터를 무제한으로 이용 가능하며 T멤버십 VIP, 구독서비스 할인 등 다양한 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['5G generation', 'LTE generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 60.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '(구) T플랜', '영문상품명': 'Data Infinity', '월정액': 100000, '상품코드매핑': ['NA00005959'], '부가세제외월정액': 90909, '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['구버전티플랜', '구버전T플랜', '데이터무제한', '데이터공유', '가족모아', 'T가족모아데이터', '가족끼리데이터공유', '데이터공유요금제', 'VIP멤버십제공', '데이터공유가능', '스마트워치요금무료', '태블릿요금무료', '로밍혜택', 'OTT콘텐츠혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': 'Data 인피니티', '선택약정할인포함부가세제외월정액': 74975, 'net가격': 90909, '운영상태': '가입중단', '고유ID': 'PA00000036', '상품설명': '데이터를 무제한으로 제공하며 가족 간 데이터 공유가 가능한 T가족모아데이터를 이용할 수 있고 인피니티 만의 특화 혜택을 제공하는 요금제', '통신규격': ['3G generation', 'LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '5GX플랜', '영문상품명': '(구)5GX프라임', '상품코드매핑': ['NA00006404'], '월정액': 89000, '부가세제외월정액': 80909, '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['wavve혜택', '5GX플랜', 'Wavve', '스트리밍', 'OTT혜택', '혜택요금제', 'FLO70%할인', 'wavve70%할인', '우주패스5천원할인', '태블릿요금무료혜택', '스마트워치요금무료혜택', 'VIP멤버십혜택', 'FLO할인혜택', '우주패스할인혜택', '제휴할인혜택', '데이터무제한'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '(구)5GX프라임', '선택약정할인포함부가세제외월정액': 66725, 'net가격': 86364, '운영상태': '운영', '고유ID': 'PA00000041', '상품설명': '무제한 데이터와 T멤버십 VIP, 구독서비스 할인 등을 제공하는 혜택 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 60.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '5GX플랜', '영문상품명': '(구)5GX플래티넘', '상품코드매핑': ['NA00006405'], '월정액': 125000, '부가세제외월정액': 113637, '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['FLO무료혜택', '우주패스무료혜택', '제휴할인혜택', '데이터무제한', '혜택요금제', '최다혜택요금제', '스트리밍', 'Wavve', 'VIP멤버십혜택', '스마트워치요금무료혜택', '태블릿요금무료혜택', '5GX플랜', 'wavve혜택', '혜택요금제', 'OTT혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '(구)5GX플래티넘', '선택약정할인포함부가세제외월정액': 93705, 'net가격': 113637, '고유ID': 'PA00000042', '운영상태': '운영', '상품설명': '무제한 데이터와 T멤버십 VIP, 구독서비스 무료 등을 제공하는 혜택 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 120.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': 'T플랜', '영문상품명': 'T Plan Max', '상품코드매핑': ['NA00006539'], '월정액': 100000, '부가세제외월정액': 90909, '청구방법': '후불', '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '마케팅키워드': ['T플랜', '가족모아데이터혜택', '데이터무제한', 'T멤버십VIP혜택', 'Wavve', 'FLO', 'wavve무료', 'FLO무료', '가족', '가족간데이터공유혜택요금제', '가족공유혜택', '태블릿요금무료', '스마트워치요금무료', '콘텐츠무료', '인피니티'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': 'T플랜 맥스', '선택약정할인포함부가세제외월정액': 74975, 'net가격': 90909, '운영상태': '운영', '고유ID': 'PA00000048', '상품설명': '데이터를 무제한으로 제공하며 가족 간 데이터 공유가 가능한 T가족모아데이터를 이용할 수 있고 T멤버십 VIP, 구독서비스 무료 등 혜택을 제공하는 요금제', '통신규격': ['5G generation', 'LTE generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '5GX플랜', '영문상품명': '5GX Platinum', '상품코드매핑': ['NA00007789'], '월정액': 125000, '부가세제외월정액': 113637, '청구방법': '후불', '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '마케팅키워드': ['OTT혜택', '혜택요금제', 'wavve혜택', '5GX플랜', 'Wavve', '스트리밍', '데이터무제한', '혜택요금제', '최다혜택요금제', '태블릿요금무료혜택', '스마트워치요금무료혜택', 'VIP멤버십혜택', 'FLO무료혜택', '우주패스무료혜택', '제휴할인혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '5GX 플래티넘', '선택약정할인포함부가세제외월정액': 93705, 'net가격': 113637, '운영상태': '운영', '고유ID': 'PA00000060', '상품설명': '무제한 데이터와 T멤버십 VIP, 구독서비스 무료 등을 제공하는 혜택 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 120.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '5GX플랜', '영문상품명': '5GX Prime', '상품코드매핑': ['NA00007790'], '월정액': 89000, '부가세제외월정액': 80909, '청구방법': '후불', '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '마케팅키워드': ['Wavve', '스트리밍', 'OTT혜택', '혜택요금제', 'wavve혜택', '5GX플랜', '태블릿요금무료혜택', 'FLO할인혜택', '우주패스할인혜택', '제휴할인혜택', '데이터무제한', '혜택요금제', 'FLO70%할인', 'wavve70%할인', '우주패스5천원할인', '스마트워치요금무료혜택', 'VIP멤버십혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '5GX 프라임', '선택약정할인포함부가세제외월정액': 66725, 'net가격': 80909, '운영상태': '운영', '고유ID': 'PA00000061', '상품설명': '무제한 데이터와 T멤버십 VIP, 구독서비스 할인 등을 제공하는 혜택 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 60.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '0청년', '영문상품명': '0 Youth 89', '상품코드매핑': ['NA00008140'], '월정액': 89000, '부가세제외월정액': 80909, '청구방법': '후불', '상품가입조건': '만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)', '마케팅키워드': ['커피50%', 'wavve할인혜택', 'FLO할인혜택', '우주패스할인혜택', '멤버십VIP', '혜택요금제', '콘텐츠할인', '만34세이하', '커피', '영화', '로밍', '로밍쿠폰', '로밍할인', '커피쿠폰', '커피할인', '영화50%', '영화쿠폰', '영화할인', 'baro50%', '데이터무제한'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '0 청년 89', '선택약정할인포함부가세제외월정액': 66725, 'net가격': 80909, '고유ID': 'PA00000065', '운영상태': '운영', '상품설명': '무제한 데이터와 0청년 특화 혜택을 제공하는 만 34세 이하 개인 고객만 가입 가능한 전용 요금제', '통신규격': ['5G generation', 'LTE generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '넷플릭스 요금제', '영문상품명': '5GX Prime Plus(Netflix)', '상품코드매핑': ['NA00008721'], '월정액': 99000, '부가세제외월정액': 90000, '청구방법': '후불', '상품가입조건': '19세 이상 성인, 고객 명의당 1개만 가입 가능', '마케팅키워드': ['넷플릭스', '우주패스Netflix', 'Netflix', '넷플릭스할인', '5GX플랜', '혜택요금제', 'wavve혜택', 'OTT혜택', '넷플릭스혜택', 'Wavve', '스트리밍', '태블릿요금무료혜택', '스마트워치요금무료혜택', 'VIP멤버십혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '5GX 프라임플러스(넷플릭스)', '선택약정할인포함부가세제외월정액': 74250, 'net가격': 90000, '고유ID': 'PA00000066', '운영상태': '운영', '상품설명': '무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '넷플릭스 요금제', '월정액': 89000, '상품코드매핑': ['NA00008722'], '영문상품명': '5GX Prime(Netflix)', '부가세제외월정액': 80909, '청구방법': '후불', '상품가입조건': '19세 이상 성인, 고객 명의당 1개만 가입 가능', '마케팅키워드': ['Wavve', '스트리밍', 'Netflix', '넷플릭스', '우주패스Netflix', '혜택요금제', 'wavve혜택', 'OTT혜택', '넷플릭스혜택', '태블릿요금할인혜택', '스마트워치요금할인혜택', '콘텐츠할인', '우주패스할인', '5GX플랜', 'FLO할인', '넷플릭스할인', 'VIP멤버십혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '5GX 프라임(넷플릭스)', '선택약정할인포함부가세제외월정액': 66725, 'net가격': 80909, '운영상태': '운영', '고유ID': 'PA00000067', '상품설명': '무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제', '통신규격': ['5G generation', 'LTE generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 60.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '0청년 넷플릭스 요금제', '영문상품명': '0 Youth 99(Netflix)', '상품코드매핑': ['NA00008727'], '월정액': 99000, '부가세제외월정액': 90000, '청구방법': '후불', '상품가입조건': '만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)', '마케팅키워드': ['로밍', '데이터무제한', '커피', '영화', '만34세이하', '커피쿠폰', '커피50%', '커피', '로밍', '로밍쿠폰', '영화할인', 'baro50%', '영화', '영화쿠폰', '커피할인', '영화50%', '로밍할인', '우주패스Netflix', 'Netflix', 'wavve혜택', '넷플릭스혜택', '혜택요금제', '스트리밍', 'OTT혜택', '넷플릭스', 'Wavve'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '0 청년 99(넷플릭스)', '선택약정할인포함부가세제외월정액': 74250, 'net가격': 90000, '운영상태': '운영', '고유ID': 'PA00000068', '상품설명': '무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 넷플릭스 전용 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 100.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '0청년 넷플릭스 요금제', '영문상품명': '0 Youth 89(Netflix)', '상품코드매핑': ['NA00008728'], '월정액': 89000, '부가세제외월정액': 80909, '청구방법': '후불', '상품가입조건': '만 19세 이상~34세 이하 고객 가입 가능(고객당 1번만 가입 가능)', '마케팅키워드': ['데이터무제한', '커피50%', '영화', '로밍', '만34세이하', '커피', '영화50%', '영화', '커피쿠폰', '커피할인', '커피', '로밍쿠폰', 'baro50%', '로밍', '영화쿠폰', '영화할인', 'Wavve', '스트리밍', 'Netflix', '넷플릭스', '로밍할인', '우주패스Netflix', '혜택요금제', 'wavve혜택', 'OTT혜택', '넷플릭스혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '0 청년 89(넷플릭스)', '선택약정할인포함부가세제외월정액': 66725, 'net가격': 80909, '운영상태': '운영', '고유ID': 'PA00000069', '상품설명': '무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 만 19세 이상 34세 이하 개인고객만 가입 가능한 넷플릭스 전용 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '넷플릭스 요금제', '영문상품명': '5GX Premium(Netflix)', '상품코드매핑': ['NA00008720'], '월정액': 109000, '부가세제외월정액': 99091, '청구방법': '후불', '상품가입조건': '19세 이상 성인, 고객 명의당 1개만 가입 가능', '마케팅키워드': ['Netflix', '넷플릭스', '우주패스Netflix', 'OTT혜택', '넷플릭스혜택', 'Wavve', '스트리밍', '스마트워치요금무료혜택', 'VIP멤버십혜택', '5GX플랜', '태블릿요금무료혜택', 'wavve혜택', '넷플릭스무료', '혜택요금제'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '5GX 프리미엄(넷플릭스)', '선택약정할인포함부가세제외월정액': 81720, 'net가격': 99091, '고유ID': 'PA00000071', '운영상태': '운영', '상품설명': '무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 100.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '다이렉트 넷플릭스 요금제', '영문상품명': 'Direct 5G 69(Netflix)', '상품코드매핑': ['NA00008724'], '월정액': 69000, '부가세제외월정액': 62728, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['온라인', '비대면', '데이터무제한', 'T다샵', '우주패스Netflix', '티다이렉트샵', '티다샵', 'USIM개통', '유심개통', '자급제', '쓰던폰', 'Netflix', '넷플릭스', 'Wavve', '다이렉트 넷플릭스', 'T다이렉트전용요금제', 'wavve혜택', '온라인전용요금제', '넷플릭스혜택', '혜택요금제', '스트리밍', 'OTT혜택', '태블릿요금무료혜택', '스마트워치요금무료혜택', '넷플릭스할인', 'VIP멤버십혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '다이렉트5G 69(넷플릭스)', '선택약정할인포함부가세제외월정액': 69000, 'net가격': 62728, '고유ID': 'PA00000072', '운영상태': '운영', '상품설명': '무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['5G generation', 'LTE generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '다이렉트 넷플릭스 요금제', '영문상품명': 'Direct 5G 62(Netflix)', '상품코드매핑': ['NA00008725'], '월정액': 62000, '부가세제외월정액': 56364, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['온라인', '티다샵', '태블릿요금무료혜택', '콘텐츠할인', '자급제', '혜택요금제', '티다이렉트샵', '비대면', '데이터무제한', '다이렉트 넷플릭스', '넷플릭스혜택', '쓰던폰', '스트리밍', '스마트워치요금무료혜택', '유심개통', '우주패스할인', '우주패스Netflix', '온라인전용요금제', 'T다샵', 'OTT혜택', 'Netflix', 'FLO할인', 'Wavve', 'VIP멤버십혜택', 'USIM개통', 'T다이렉트전용요금제', '넷플릭스할인', '넷플릭스', 'wavve혜택'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '다이렉트5G 62(넷플릭스)', '선택약정할인포함부가세제외월정액': 62000, 'net가격': 56364, '고유ID': 'PA00000073', '운영상태': '운영', '상품설명': '무제한 데이터와 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['LTE generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 60.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '0청년 다이렉트 넷플릭스 요금제', '영문상품명': '0 Youth Direct 69(Netflix)', '상품코드매핑': ['NA00008730'], '월정액': 69000, '부가세제외월정액': 62728, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 34세 이하 개인 고객 가입 가능(고객당 1번만 가입 가능)', '청구방법': '후불', '마케팅키워드': ['비대면', '온라인', '티다샵', 'T다샵', '유심개통', '티다이렉트샵', '쓰던폰', 'USIM개통', '데이터무제한', '자급제', '우주패스Netflix', 'Netflix', '넷플릭스', '온라인전용요금제', '다이렉트 넷플릭스', '혜택요금제', 'wavve혜택', 'OTT혜택', '넷플릭스혜택', 'Wavve', '스트리밍', 'T다이렉트전용요금제', '넷플릭스할인', '만34세이하', '커피할인', '영화50%', '커피', '커피쿠폰', '로밍혜택', '커피50%', '커피혜택', '영화혜택', '영화쿠폰', '영화할인', '영화', '로밍쿠폰', '로밍할인', 'baro50%', '로밍'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '0 청년 다이렉트 69(넷플릭스)', '선택약정할인포함부가세제외월정액': 69000, 'net가격': 62728, '고유ID': 'PA00000074', '운영상태': '운영', '상품설명': '무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 19세 이상 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 100.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '0청년 다이렉트 넷플릭스 요금제', '영문상품명': '0 Youth Direct 62(Netflix)', '상품코드매핑': ['NA00008731'], '월정액': 62000, '부가세제외월정액': 56364, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 19세 이상 34세 이하 개인 고객 가입 가능(고객당 1번만 가입 가능)', '청구방법': '후불', '마케팅키워드': ['데이터무제한', '온라인', '비대면', 'USIM개통', '유심개통', '자급제', '쓰던폰', 'Netflix', '넷플릭스', 'T다샵', '우주패스Netflix', '티다이렉트샵', '티다샵', '넷플릭스혜택', '혜택요금제', '스트리밍', 'OTT혜택', 'Wavve', '넷플릭스할인', '다이렉트 넷플릭스', 'T다이렉트전용요금제', 'wavve혜택', '온라인전용요금제', '커피50%', '커피', '영화혜택', '로밍혜택', '만34세이하', '커피혜택', '영화50%', '영화', '커피쿠폰', '커피할인', '로밍할인', '로밍', '로밍쿠폰', '영화할인', 'baro50%', '영화쿠폰'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '0 청년 다이렉트 62(넷플릭스)', '선택약정할인포함부가세제외월정액': 62000, 'net가격': 56364, '고유ID': 'PA00000075', '운영상태': '운영', '상품설명': '무제한 데이터와 0청년 특화 혜택 외 넷플릭스 멤버십을 제공하는 넷플릭스 전용 요금제로 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서 만 19세 이상 34세 이하 개인 고객만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 80.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '0청년', '영문상품명': '0 Youth 109', '상품코드매핑': ['NA00008676'], '월정액': 109000, '부가세제외월정액': 99091, '청구방법': '후불', '상품가입조건': '만 34세 이하 고객 가입 가능(고객당 1번만 가입 가능)', '마케팅키워드': ['로밍', '로밍쿠폰', '로밍할인', '만34세이하', '멤버십VIP', '스마트워치요금무료', '영화', 'YoutubePremium혜택', '단말보험할인', '데이터무제한', '5GX플랜', 'baro50%', '영화쿠폰', '영화할인', '유튜브무료제공요금제', '유튜브요금제', '유튜브혜택', '커피', '커피50%', '커피쿠폰', '영화50%', '태블릿요금무료', '혜택요금제', '커피할인'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '0 청년 109', '선택약정할인포함부가세제외월정액': 81720, 'net가격': 99091, '운영상태': '운영', '고유ID': 'PA00000076', '상품설명': '무제한 데이터와 0청년 특화 혜택 외 유튜브 프리미엄을 제공하는 만 34세 이하 개인 고객만 가입 가능한 유튜브 혜택 특화 전용 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 120.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}, {'y': {'라인업': '다이렉트플랜', '영문상품명': 'Direct5G 76', '월정액': 76000, '상품코드매핑': ['NA00008685'], '부가세제외월정액': 69091, '상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['온라인', '티다이렉트샵', '티다샵', 'USIM개통', '유심개통', '자급제', '쓰던폰', '비대면', '데이터무제한', 'T다샵', '다이렉트플랜', '온라인전용요금제', '스마트워치기기할인', '태블릿기기할인', '유튜브무료제공요금제', '혜택요금제', '유튜브혜택', 'YoutubePremium혜택', 'T다이렉트전용요금제', '유튜브요금제', '스마트폰기기할인'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '다이렉트5G 76', '선택약정할인포함부가세제외월정액': 76000, 'net가격': 69091, '운영상태': '운영', '고유ID': 'PA00000077', '상품설명': '데이터를 무제한으로 이용 가능하며 유튜브 프리미엄 혜택 및 T멤버십 VIP, 구독서비스 무료 등 다양한 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한 온라인 전용 무약정 요금제', '통신규격': ['LTE generation', '5G generation']}, 'd': {'최대데이터선물가능용량': 2.0, '데이터소진후데이터제공속도': 0.0, '기본제공데이터중mvoip용량': 99999.0, '데이터리필가능용량': 99999.0, '기본제공데이터중공유가능용량': 100.0, '데이터소진후최대금액및속도제한적용': 'N', '데이터리필쿠폰선물가능여부': 'Y', '데이터선물받기가능여부': 'Y', '시니어대상데이터소진후최대금액및속도제한적용': 'N', '기본제공데이터용량': 99999.0}}]\n    ", "summary_text": "당신의 질문이 무제한 데이터 요금제에 관한 것이라면, 다양한 요금제 옵션 중에서 선택할 수 있도록 상세한 정보를 제공하겠습니다. 이 정보는 요금제의 특징, 가격, 가입 조건 등으로 구성되어 있습니다.\n\n### 추천 무제한 데이터 요금제\n\n1. **다이렉트5G 69**\n   - **월정액**: 69,000원 (부가세 제외: 62,728원)\n   - **가입 조건**: T다이렉트샵(SK텔레콤 공식 온라인 채널)을 통해 신규/기변한 고객만 가입 가능\n   - **상품 설명**: 무제한 데이터 이용이 가능하며, T멤버십 VIP, 구독 서비스 무료 등 여러 혜택을 제공합니다. 온라인 전용 무약정 요금제입니다.\n   - **기타 특징**:\n     - 데이터 소진 후 속도: 0.0 Mbps\n     - 추가 데이터 선물 가능량: 최대 2.0 GB 가능\n\n2. **다이렉트5G 62**\n   - **월정액**: 62,000원 (부가세 제외: 56,364원)\n   - **가입 조건**: T다이렉트샵을 통해 신규/기변한 고객 가입 가능\n   - **상품 설명**: 무제한 데이터 제공하며 T멤버십 VIP, 구독 서비스 할인 포함.\n  \n3. **Data Infinity**\n   - **월정액**: 100,000원 (부가세 제외: 90,909원)\n   - **가입 조건**: 5G/LTE 휴대폰 이용 고객 가입 가능\n   - **상품 설명**: 수량·속도 제한 없는 무제한 데이터와 가족 간 데이터 공유 가능. 그러나 현재 가입 중단 상태입니다.\n\n4. **5GX Prime (구버전)**\n   - **월정액**: 89,000원 (부가세 제외: 80,909원)\n   - **가입 조건**: 5G/LTE 휴대폰 이용 고객 가입 가능\n   - **상품 설명**: 무제한 데이터 제공 및 다양한 특별 혜택 포함.\n\n5. **0 청년 89 (넷플릭스 포함)**\n   - **월정액**: 89,000원 (부가세 제외: 80,909원)\n   - **가입 조건**: 만 34세 이하 고객만 가입 가능\n   - **상품 설명**: 무제한 데이터 이용과 다양한 청년 혜택 제공.\n\n### 종합 요약 및 추천\n위의 요금제 중 가장 추천하는 것은 **다이렉트5G 69 요금제**입니다. 이 요금제는 무제한 데이터와 다양한 혜택을 제공하며, 월정액도 합리적입니다. 만약 34세 이하라면 **0 청년 89** 요금제는 추가 혜택과 함께 매우 매력적입니다. \n\n무제한 데이터 요금제를 선택할 때에는 본인의 사용 패턴과 필요에 따라 요금제를 잘 비교하고 선택하는 것이 중요합니다. 여러 요금제를 둘러본 후, 개인 상황에 맞는 최적의 요금제를 선택하시길 권장합니다. 추가 질문이나 도움이 필요하시면 언제든 문의해주세요."}

In [100]:
dd.keys()

dict_keys(['user_query', 'cypher_query', 'records', 'summary_prompt', 'summary_text'])

In [95]:
print(dd['summary_prompt'])


        아래에 Neo4j 쿼리 결과 샘플 20건이 있습니다.
        이 샘플을 참고해 전체 데이터의 특징과 예시를 요약하세요.
        
        (전체 데이터가 너무 커서 일부만 제공됩니다.)
        
        User query:
        무제한 데이터 요금제의 특징, 가격 및 가입 방법에 대한 정보 요청
        
        Cypher Query:
        MATCH (y:요금제)-[:제공]->(d:데이터용량)
WHERE y.상품설명 CONTAINS '무제한' OR y.마케팅키워드 CONTAINS '무제한'
RETURN y.상품명, y.부가세제외월정액, y.상품가입조건, d.기본제공데이터용량
        
        Query Result Sample:
        [{'y.상품명': '다이렉트5G 69', 'y.부가세제외월정액': 62728, 'y.상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능', 'd.기본제공데이터용량': 99999.0}, {'y.상품명': '다이렉트5G 62', 'y.부가세제외월정액': 56364, 'y.상품가입조건': 'T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능', 'd.기본제공데이터용량': 99999.0}, {'y.상품명': 'Data 인피니티', 'y.부가세제외월정액': 90909, 'y.상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', 'd.기본제공데이터용량': 99999.0}, {'y.상품명': '(구)5GX프라임', 'y.부가세제외월정액': 80909, 'y.상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', 'd.기본제공데이터용량': 99999.0}, {'y.상품명': '(구)5GX플래티넘', 'y.부가세제외월정액': 113637, 'y.상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', 'd.기본제공데이터용량': 99999.0}, {'y.

In [5]:
print(paraphrase_intent("요금제에 음악 듣기 서비스가 되는 요금제가 있나요"))

{
  "intent": "음악 듣기 서비스가 포함된 요금제 문의",
  "paraphrased_query": "음악 듣기 서비스가 포함된 요금제를 알고 싶습니다.",
  "expected_answer": "음악 듣기 서비스가 포함된 요금제 목록"
}


In [6]:
print(paraphrase_intent("다이렉트랑 일반 요금제 차이가 뭐야?"))

{
  "intent": "다이렉트 요금제와 일반 요금제의 차이 이해",
  "paraphrased_query": "다이렉트 요금제와 일반 요금제의 구체적인 차이를 알고 싶습니다.",
  "expected_answer": "다이렉트 요금제와 일반 요금제의 특징 및 차이점"
}


In [7]:
print(paraphrase_intent("데이터80기가 쓰면서 넷플릭스 쓸수 있는 요금제는"))

{
  "intent": "넷플릭스 사용을 위한 적합한 요금제 문의",
  "paraphrased_query": "넷플릭스를 사용하면서 80GB의 데이터를 제공하는 요금제를 알고 싶습니다.",
  "expected_answer": "고객이 넷플릭스를 사용할 수 있는 80GB 데이터 요금제 목록"
}


In [9]:
import json

In [10]:
query_metadata = json.loads(paraphrase_intent("데이터80기가 쓰면서 넷플릭스 쓸수 있는 요금제는"))

In [11]:
steps = plan_steps(query_metadata)

In [12]:
steps

['1. 사용자 요구사항을 분석합니다.',
 '2. 넷플릭스를 사용하기 위한 데이터 요금제의 기준을 정의합니다.',
 '3. 80기가 이상의 데이터를 제공하는 요금제 목록을 수집합니다.',
 '4. 각 요금제가 넷플릭스를 포함하는지 확인합니다.',
 '5. 결과를 자연어로 요약합니다.',
 '6. 최종 답변의 정확성을 검토합니다.']

In [107]:
metadata

{'intent': '무제한 데이터 요금제에 대한 정보 요청',
 'paraphrased_query': '무제한 데이터 요금제에 대해 알고 싶습니다. (추가 고려사항: 쿼리 결과를 바탕으로 보다 명확한 형태의 추천 및 요약을 제공해야 합니다.) (추가 고려사항: 요금제 목록 뒤에 추천 및 결론을 명확하게 추가하여 요약하고, 특정 요금제를 추천하는 내용을 포함해야 합니다.)',
 'expected_answer': '무제한 데이터 요금제의 자세한 내용 및 조건'}

In [39]:
result = search_neo4j("무제한 데이터 요금제")

In [40]:
result

{'user_query': '무제한 데이터 요금제',
 'cypher_query': "MATCH (y:요금제)\nWHERE y.상품설명 CONTAINS '무제한 데이터'\nRETURN y",
 'raw_records': [{'y': {'라인업': '5GX플랜',
    '영문상품명': '(구)5GX프라임',
    '상품코드매핑': ['NA00006404'],
    '월정액': 89000,
    '부가세제외월정액': 80909,
    '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능',
    '청구방법': '후불',
    '마케팅키워드': ['wavve혜택',
     '5GX플랜',
     'Wavve',
     '스트리밍',
     'OTT혜택',
     '혜택요금제',
     'FLO70%할인',
     'wavve70%할인',
     '우주패스5천원할인',
     '태블릿요금무료혜택',
     '스마트워치요금무료혜택',
     'VIP멤버십혜택',
     'FLO할인혜택',
     '우주패스할인혜택',
     '제휴할인혜택',
     '데이터무제한'],
    '상품분류': '상품 > 기본요금제 > 휴대폰 요금제',
    '상품명': '(구)5GX프라임',
    '선택약정할인포함부가세제외월정액': 66725,
    'net가격': 86364,
    '운영상태': '운영',
    '고유ID': 'PA00000041',
    '상품설명': '무제한 데이터와 T멤버십 VIP, 구독서비스 할인 등을 제공하는 혜택 요금제',
    '통신규격': ['LTE generation', '5G generation']}},
  {'y': {'라인업': '5GX플랜',
    '영문상품명': '(구)5GX플래티넘',
    '상품코드매핑': ['NA00006405'],
    '월정액': 125000,
    '부가세제외월정액': 113637,
    '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능',
 

In [61]:
def clean_json_text(raw_text: str) -> dict:
    """
    LLM 출력에서 ```json ... ``` 코드 블록 제거 후 JSON 파싱
    """
    # 1. 코드블록 제거
    cleaned = re.sub(r"^```json\s*", "", raw_text.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r"```$", "", cleaned.strip(), flags=re.MULTILINE)
    
    # 2. JSON 파싱
    return json.loads(cleaned)

def validate_cypher_step(user_intent: str, cypher_query: str, raw_records: str):
    prompt = f"""
        당신은 Neo4j 검색 에이전트의 품질 검수자입니다.
        아래 정보에 기반하여 Cypher 쿼리 생성 및 실행이 적절했는지 평가하세요.
        
        평가 기준:
        1. 생성된 쿼리가 사용자의 의도를 올바르게 반영했는가?
        2. 쿼리 실행 결과가 의도에 부합하는 데이터를 반환했는가?
        3. 쿼리 구문에 문제가 없었는가?
        
        아래 정보를 참고하세요:
        - 사용자 의도:
        {user_intent}
        
        - 생성된 Cypher 쿼리:
        {cypher_query}
        
        - 쿼리 결과:
        {raw_records}
        
        아래 형식으로 JSON을 출력하세요:
        
        {{
          "is_valid": "Yes" or "No",
          "reason": "간단하고 명확한 문제 여부 설명",
          "suggestion": "문제가 있다면 개선 방안, 없다면 'None'"
        }}
    """
    # print(prompt)
    response = llm.invoke([
        {"role": "system", "content": "You are an expert validator of Cypher queries."},
        {"role": "user", "content": prompt}
    ])
    parsed = clean_json_text(response.content.strip())
    return parsed

In [43]:
paraphrased = paraphrase_intent("무제한 데이터 요금제")

In [46]:
str(json.loads(paraphrased))

"{'intent': '무제한 데이터 요금제에 대한 정보 요청', 'paraphrased_query': '무제한 데이터 요금제의 상세 내용과 가격을 알고 싶습니다.', 'expected_answer': '무제한 데이터 요금제의 가격, 조건 및 특징에 대한 정보'}"

In [62]:
resp = validate_cypher_step(str(json.loads(paraphrased)), result['cypher_query'], result['raw_records'])


        당신은 Neo4j 검색 에이전트의 품질 검수자입니다.
        아래 정보에 기반하여 Cypher 쿼리 생성 및 실행이 적절했는지 평가하세요.

        평가 기준:
        1. 생성된 쿼리가 사용자의 의도를 올바르게 반영했는가?
        2. 쿼리 실행 결과가 의도에 부합하는 데이터를 반환했는가?
        3. 쿼리 구문에 문제가 없었는가?

        아래 정보를 참고하세요:
        - 사용자 의도:
        {'intent': '무제한 데이터 요금제에 대한 정보 요청', 'paraphrased_query': '무제한 데이터 요금제의 상세 내용과 가격을 알고 싶습니다.', 'expected_answer': '무제한 데이터 요금제의 가격, 조건 및 특징에 대한 정보'}

        - 생성된 Cypher 쿼리:
        MATCH (y:요금제)
WHERE y.상품설명 CONTAINS '무제한 데이터'
RETURN y

        - 쿼리 결과:
        [{'y': {'라인업': '5GX플랜', '영문상품명': '(구)5GX프라임', '상품코드매핑': ['NA00006404'], '월정액': 89000, '부가세제외월정액': 80909, '상품가입조건': '5G/LTE 휴대폰 이용 고객 가입 가능', '청구방법': '후불', '마케팅키워드': ['wavve혜택', '5GX플랜', 'Wavve', '스트리밍', 'OTT혜택', '혜택요금제', 'FLO70%할인', 'wavve70%할인', '우주패스5천원할인', '태블릿요금무료혜택', '스마트워치요금무료혜택', 'VIP멤버십혜택', 'FLO할인혜택', '우주패스할인혜택', '제휴할인혜택', '데이터무제한'], '상품분류': '상품 > 기본요금제 > 휴대폰 요금제', '상품명': '(구)5GX프라임', '선택약정할인포함부가세제외월정액': 66725, 'net가격': 86364, '운영상태': '운영', '고유ID': 'PA00000041

In [58]:
import ast

In [63]:
resp

{'is_valid': 'Yes',
 'reason': '생성된 쿼리가 사용자의 의도를 올바르게 반영하고 있으며, 실행 결과도 기대한 데이터(무제한 데이터 요금제 가격, 조건, 특징 등)를 반환했습니다.',
 'suggestion': 'None'}

In [ ]:
def validate_summary_step(user_intent: str, raw_records: str, summary_text: str):
    prompt = f"""
        당신은 고객 응대 AI의 품질 검수자입니다.
        아래 정보에 기반하여 최종 요약 답변이 적절했는지 평가하세요.
        
        평가 기준:
        1. 요약 답변이 사용자의 의도에 부합하는가?
        2. 정보가 누락되거나 과장되지 않았는가?
        3. 사용자 입장에서 이해하기 쉬운가?
        
        아래 정보를 참고하세요:
        - 사용자 의도:
        {user_intent}
        
        - 쿼리 결과:
        {raw_records}
        
        - 요약 답변:
        {summary_text}
        
        아래 형식으로 JSON을 출력하세요:
        
        {{
          "is_valid": "Yes" or "No",
          "reason": "간단하고 명확한 문제 여부 설명",
          "suggestion": "문제가 있다면 개선 방안, 없다면 'None'"
        }}
    """
    response = llm.invoke([
        {"role": "system", "content": "You are an expert validator of AI-generated summaries."},
        {"role": "user", "content": prompt}
    ])
    return response.content.strip()

In [ ]:
def agent_loop(user_input: str, max_attempts: int = 3):
    attempt = 1
    while attempt <= max_attempts:
        print(f"\n[Attempt {attempt}]")
        steps = plan_steps_llm(user_input)

        print("\n[Executing LangChain GraphCypherQAChain]")
        answer = qa_chain.run(user_input)

        print("\n[Answer Generated]")
        print(answer)

        if validate_answer(user_input, answer):
            print("\n✅ 답변이 적절합니다.")
            print("최종 답변:", answer)
            return answer
        else:
            print("\n⚠️ 답변이 부적절합니다. Replanning...")
            attempt += 1

    print("\n❌ 최대 시도 횟수를 초과했습니다.")